In [17]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier 


pd.set_option('display.max_columns', None)   # 모든 컬럼 표시
pd.set_option('display.width', None)         # 줄바꿈 없이 전체 폭 사용
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 생략 안 함print(df)

df = pd.read_csv('data/reviews_joined_sample20k.csv')
df.head()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,developer_response,timestamp_dev_responded,primarily_steam_deck,appid_1,game_name,genre,rnd
0,236390,214935645,76561199408691293,70,8,6553,199,6528,NaN,1767374825,english,I gave my soul to the Snail,1767374170,1767374170,True,0,0,0.50000,0,False,False,False,NaN,NaN,False,236390,War Thunder,"['Action', 'Massively Multiplayer', 'Simulation', 'Free To Play']",6.626704e-07
1,381210,210513031,76561199034590636,0,9,23353,1308,16597,NaN,1767470513,polish,You will love and hate this game at the same time,1764082672,1764082672,True,0,0,0.50000,0,True,False,False,NaN,NaN,False,381210,Dead by Daylight,['Action'],2.654878e-06
2,1551360,205317346,76561199062613729,71,2,4780,0,4224,NaN,1761669035,turkish,müthiş,1758979826,1758979826,True,0,0,0.50000,0,True,False,False,NaN,NaN,False,1551360,Forza Horizon 5,"['Action', 'Adventure', 'Racing', 'Simulation', 'Sports']",3.815672e-06
3,381210,201395452,76561198998550758,7,2,52724,119,41315,NaN,1766683805,english,"The community is absolute dogshit.\nFucking insane falloff since 2020.\nAvoid at all cost, just like majority of asymetrical games.\n🤮",1754259767,1754259767,False,1,0,0.52381,0,True,False,False,NaN,NaN,False,381210,Dead by Daylight,['Action'],5.021407e-06
4,236390,205324525,76561199588258066,0,1,604,0,218,NaN,1766288019,english,the snail,1758985739,1758985739,True,0,0,0.50000,0,False,True,False,NaN,NaN,False,236390,War Thunder,"['Action', 'Massively Multiplayer', 'Simulation', 'Free To Play']",5.237950e-06


| 한글 컬럼명 | 영문 컬럼명 | 설명(내용) | 범위(실제 분포 기준) |
|---|---|---|---|
| 앱 ID | app_id | Steam 앱 고유 식별자 | 정수 (수천만 단위, 예: 43 ~ 85,000,000+) |
| 앱 이름 | app_name | 게임 또는 앱 이름 | 문자열 |
| 리뷰 ID | review_id | 리뷰 고유 식별자 | 정수 (고유값) |
| 리뷰 언어 | language | 리뷰가 작성된 언어 | 문자열 (예: english, schinese 등) |
| 리뷰 내용 | review | 리뷰 텍스트 본문 | 문자열 (길이 가변) |
| 리뷰 생성 시각 | timestamp_created | 리뷰 작성 시각 | Unix Timestamp (약 1.29B ~ 1.61B) |
| 리뷰 수정 시각 | timestamp_updated | 리뷰 마지막 수정 시각 | Unix Timestamp (약 1.29B ~ 2.28B) |
| 추천 여부 | recommended | 게임 추천 여부 | Boolean (true / false, true ≈ 87%) |
| 도움됨 투표 수 | votes_helpful | 도움됨(Helpful) 투표 수 | 정수 (0 ~ 약 4,900) |
| 재미있음 투표 수 | votes_funny | 재미있음(Funny) 투표 수 | 정수 (0 ~ 약 27,000) |
| 가중 투표 점수 | weighted_vote_score | 도움됨 기반 가중 점수 | 실수 (0.0 ~ 1.0) |
| 댓글 수 | comment_count | 리뷰 댓글 수 | 정수 (0 ~ 약 1,300,000) |
| 스팀 구매 여부 | steam_purchase | Steam에서 직접 구매했는지 여부 | Boolean (true ≈ 77%) |
| 무료 획득 여부 | received_for_free | 무료 획득 여부 | Boolean (true ≈ 3%) |
| 얼리액세스 리뷰 | written_during_early_access | 얼리 액세스 중 작성 여부 | Boolean (true ≈ 9%) |
| 작성자 SteamID | author.steamid | 리뷰 작성자 SteamID | 64-bit 정수 (약 7.6e16 ~ 7.7e16) |
| 작성자 보유 게임 수 | author.num_games_owned | 보유 게임 개수 | 정수 (0 ~ 약 21,700,000) |
| 작성자 리뷰 수 | author.num_reviews | 작성자 전체 리뷰 수 | 정수 (0 ~ 약 1,290,000) |
| 누적 플레이 시간 | author.playtime_forever | 총 플레이 시간 | 초 단위 정수 (0 ~ 약 85,000,000초) |
| 최근 2주 플레이 시간 | author.playtime_last_two_weeks | 최근 2주 플레이 시간 | 초 단위 정수 (0 ~ 약 3,700,000초) |
| 리뷰 시점 플레이 시간 | author.playtime_at_review | 리뷰 당시 플레이 시간 | 초 단위 정수 (0 ~ 약 27,000초) |
| 마지막 플레이 시각 | author.last_played | 마지막 플레이 시각 | Unix Timestamp (약 1.29B ~ 1.61B) |


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   appid                        20000 non-null  int64  
 1   recommendationid             20000 non-null  int64  
 2   steamid                      20000 non-null  int64  
 3   num_games_owned              20000 non-null  int64  
 4   num_reviews_author           20000 non-null  int64  
 5   playtime_forever             20000 non-null  int64  
 6   playtime_last_two_weeks      20000 non-null  int64  
 7   playtime_at_review           20000 non-null  int64  
 8   deck_playtime_at_review      378 non-null    float64
 9   last_played                  20000 non-null  int64  
 10  language                     20000 non-null  object 
 11  review                       19935 non-null  object 
 12  timestamp_created            20000 non-null  int64  
 13  timestamp_update

In [19]:
df.isnull().sum()

appid                              0
recommendationid                   0
steamid                            0
num_games_owned                    0
num_reviews_author                 0
playtime_forever                   0
playtime_last_two_weeks            0
playtime_at_review                 0
deck_playtime_at_review        19622
last_played                        0
language                           0
review                            65
timestamp_created                  0
timestamp_updated                  0
voted_up                           0
votes_up                           0
votes_funny                        0
weighted_vote_score                0
comment_count                      0
steam_purchase                     0
received_for_free                  0
written_during_early_access        0
developer_response             19944
timestamp_dev_responded        19944
primarily_steam_deck               0
appid_1                            0
game_name                          0
g

In [20]:
df.describe()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,timestamp_created,timestamp_updated,votes_up,votes_funny,weighted_vote_score,comment_count,timestamp_dev_responded,appid_1,rnd
count,2.000000e+04,2.000000e+04,2.000000e+04,20000.000000,20000.000000,2.000000e+04,20000.000000,2.000000e+04,378.000000,2.000000e+04,2.000000e+04,2.000000e+04,20000.000000,20000.000000,20000.000000,20000.00000,5.600000e+01,2.000000e+04,2.000000e+04
mean,1.239968e+06,2.077497e+08,7.656120e+16,55.883600,8.752100,1.477358e+04,476.287000,1.227571e+04,1559.732804,1.762497e+09,1.760872e+09,1.760980e+09,0.633400,0.127600,0.502658,0.04080,1.759042e+09,1.239968e+06,9.613474e-03
std,8.641149e+05,4.905040e+06,6.076634e+08,189.987309,27.650139,3.523323e+04,1143.565055,3.313052e+04,3817.019513,1.148966e+07,4.758970e+06,4.740526e+06,12.127261,3.392623,0.022929,0.34791,4.683899e+06,8.641149e+05,5.523837e-03
min,4.400000e+02,1.994059e+08,7.656120e+16,0.000000,1.000000,5.000000e+00,0.000000,5.000000e+00,1.000000,1.469895e+09,1.752100e+09,1.752100e+09,0.000000,0.000000,0.294441,0.00000,1.752474e+09,4.400000e+02,6.626704e-07
25%,5.268700e+05,2.032226e+08,7.656120e+16,0.000000,1.000000,1.462000e+03,0.000000,8.020000e+02,62.000000,1.761509e+09,1.756605e+09,1.756735e+09,0.000000,0.000000,0.500000,0.00000,1.755001e+09,5.268700e+05,4.842644e-03
50%,1.172470e+06,2.079370e+08,7.656120e+16,0.000000,3.000000,4.494500e+03,0.000000,2.752000e+03,293.500000,1.765692e+09,1.761808e+09,1.762023e+09,0.000000,0.000000,0.500000,0.00000,1.758485e+09,1.172470e+06,9.624031e-03
75%,1.771300e+06,2.121995e+08,7.656120e+16,51.000000,8.000000,1.271575e+04,370.000000,9.195250e+03,1551.750000,1.767226e+09,1.764548e+09,1.764595e+09,0.000000,0.000000,0.500000,0.00000,1.764038e+09,1.771300e+06,1.442103e-02
max,3.241660e+06,2.152598e+08,7.656120e+16,7706.000000,2542.000000,1.457369e+06,17144.000000,1.396679e+06,51724.000000,1.767652e+09,1.767648e+09,1.767648e+09,1511.000000,438.000000,0.949417,14.00000,1.767615e+09,3.241660e+06,1.914195e-02


In [21]:
df.columns

Index(['appid', 'recommendationid', 'steamid', 'num_games_owned',
       'num_reviews_author', 'playtime_forever', 'playtime_last_two_weeks',
       'playtime_at_review', 'deck_playtime_at_review', 'last_played',
       'language', 'review', 'timestamp_created', 'timestamp_updated',
       'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score',
       'comment_count', 'steam_purchase', 'received_for_free',
       'written_during_early_access', 'developer_response',
       'timestamp_dev_responded', 'primarily_steam_deck', 'appid_1',
       'game_name', 'genre', 'rnd'],
      dtype='object')

In [22]:
# # 도메인으로 필터링 함
# drop_columns = [
#     'Unnamed: 0',
#     'app_id',
#     'app_name',
#     'review_id',
# ]
# df = df.drop(columns=drop_columns)
# df

In [23]:
df.count()

appid                          20000
recommendationid               20000
steamid                        20000
num_games_owned                20000
num_reviews_author             20000
playtime_forever               20000
playtime_last_two_weeks        20000
playtime_at_review             20000
deck_playtime_at_review          378
last_played                    20000
language                       20000
review                         19935
timestamp_created              20000
timestamp_updated              20000
voted_up                       20000
votes_up                       20000
votes_funny                    20000
weighted_vote_score            20000
comment_count                  20000
steam_purchase                 20000
received_for_free              20000
written_during_early_access    20000
developer_response                56
timestamp_dev_responded           56
primarily_steam_deck           20000
appid_1                        20000
game_name                      20000
g

In [24]:
review_dt = pd.to_datetime(df["timestamp_created"], unit="s")
last_dt   = pd.to_datetime(df["last_played"], unit="s")

df["days_after_review"] = (last_dt - review_dt).dt.days

df["churn"] = (df["days_after_review"] > 30).astype(int)

# 예외 처리
df.loc[df["last_played"] == 0, "churn"] = 1
df.loc[df["days_after_review"] < 0, "churn"] = 1


In [25]:
# df['days_since_last_play']


In [34]:
df[df["churn"]==1].count() 

appid                          11587
recommendationid               11587
steamid                        11587
num_games_owned                11587
num_reviews_author             11587
playtime_forever               11587
playtime_last_two_weeks        11587
playtime_at_review             11587
deck_playtime_at_review          226
last_played                    11587
language                       11587
review                         11563
timestamp_created              11587
timestamp_updated              11587
voted_up                       11587
votes_up                       11587
votes_funny                    11587
weighted_vote_score            11587
comment_count                  11587
steam_purchase                 11587
received_for_free              11587
written_during_early_access    11587
developer_response                40
timestamp_dev_responded           40
primarily_steam_deck           11587
appid_1                        11587
game_name                      11587
g

In [33]:
df

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,deck_playtime_at_review,last_played,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,developer_response,timestamp_dev_responded,primarily_steam_deck,appid_1,game_name,genre,rnd,days_after_review,churn,review_dt,last_played_dt
0,236390,214935645,76561199408691293,70,8,6553,199,6528,NaN,1767374825,english,I gave my soul to the Snail,1767374170,1767374170,True,0,0,0.50000,0,False,False,False,NaN,NaN,False,236390,War Thunder,"['Action', 'Massively Multiplayer', 'Simulation', 'Free To Play']",6.626704e-07,0,0,2026-01-02 17:16:10,2026-01-02 17:27:05
1,381210,210513031,76561199034590636,0,9,23353,1308,16597,NaN,1767470513,polish,You will love and hate this game at the same time,1764082672,1764082672,True,0,0,0.50000,0,True,False,False,NaN,NaN,False,381210,Dead by Daylight,['Action'],2.654878e-06,39,1,2025-11-25 14:57:52,2026-01-03 20:01:53
2,1551360,205317346,76561199062613729,71,2,4780,0,4224,NaN,1761669035,turkish,müthiş,1758979826,1758979826,True,0,0,0.50000,0,True,False,False,NaN,NaN,False,1551360,Forza Horizon 5,"['Action', 'Adventure', 'Racing', 'Simulation', 'Sports']",3.815672e-06,31,1,2025-09-27 13:30:26,2025-10-28 16:30:35
3,381210,201395452,76561198998550758,7,2,52724,119,41315,NaN,1766683805,english,"The community is absolute dogshit.\nFucking insane falloff since 2020.\nAvoid at all cost, just like majority of asymetrical games.\n🤮",1754259767,1754259767,False,1,0,0.52381,0,True,False,False,NaN,NaN,False,381210,Dead by Daylight,['Action'],5.021407e-06,143,1,2025-08-03 22:22:47,2025-12-25 17:30:05
4,236390,205324525,76561199588258066,0,1,604,0,218,NaN,1766288019,english,the snail,1758985739,1758985739,True,0,0,0.50000,0,False,True,False,NaN,NaN,False,236390,War Thunder,"['Action', 'Massively Multiplayer', 'Simulation', 'Free To Play']",5.237950e-06,84,1,2025-09-27 15:08:59,2025-12-21 03:33:39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,2139460,214989253,76561199805450933,0,1,5404,5404,4578,NaN,1767509031,english,amazing game\r\ngraphics are stunning\r\n love the gameplay\r\nwill get you butt whooped if you decide to go to where you don't belong,1767417387,1767417387,True,0,0,0.50000,0,False,False,False,NaN,NaN,False,2139460,Once Human,"['Action', 'Adventure', 'RPG', 'Simulation', 'Strategy', 'Free To Play']",1.913930e-02,1,0,2026-01-03 05:16:27,2026-01-04 06:43:51
19996,413150,207624891,76561197964370149,0,1,17457,0,904,NaN,1765374033,english,"family friendly, great depth, well worth £10.\r\n",1761474107,1761474107,True,0,0,0.50000,0,True,False,False,NaN,NaN,False,413150,Stardew Valley,"['Indie', 'RPG', 'Simulation']",1.913939e-02,45,1,2025-10-26 10:21:47,2025-12-10 13:40:33
19997,413150,206013726,76561199118992449,0,1,11876,0,1526,NaN,1764769196,schinese,好玩，上瘾,1759727905,1759727905,True,0,0,0.50000,0,True,False,False,NaN,NaN,False,413150,Stardew Valley,"['Indie', 'RPG', 'Simulation']",1.913994e-02,58,1,2025-10-06 05:18:25,2025-12-03 13:39:56
19998,2651280,199703559,76561198393336271,533,25,2191,0,2190,NaN,1752409648,english,"Overall, it was a fun and well-made experience. The combat mechanics and visuals are once again top-notch, and some scenes were genuinely impressive. However, it didn’t pull me in the way the first game did. While the story gets more exciting later on, I felt like the Venom arc came in too late, and Peter’s descent into darkness due to the suit could have been explored more deeply and for a longer time.The side missions were the biggest letdown for me. Many felt forced and boring. Some characters and quests also seemed like they were added just to pad out the content.Still, it was a good way to spend time. The freedom of being Spider-Man is always satisf

In [35]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# =========================
# 1) datetime + 기본 전처리
# =========================
df = df.copy()

df["review_dt"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
df = df.dropna(subset=["review_dt"]).copy()

# review_length
df["review_length"] = df["review"].fillna("").astype(str).str.len()

# deck_playtime_at_review 결측 처리 (컬럼 있으면)
if "deck_playtime_at_review" in df.columns:
    df["deck_playtime_at_review"] = df["deck_playtime_at_review"].fillna(0)

# True/False -> 0/1 정리 (LightGBM 편하게)
bool_cols = [
    "primarily_steam_deck",
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
]
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(int)

# =========================
# 2) 180일 중 마지막 90일 제외 (라벨 생성용 구간만)
# =========================
END_DATE = df["review_dt"].max()
START_DATE = END_DATE - pd.Timedelta(days=180)
LABEL_CUTOFF = END_DATE - pd.Timedelta(days=90)

df_180 = df[df["review_dt"] >= START_DATE].copy()
df_label = df_180[df_180["review_dt"] <= LABEL_CUTOFF].copy()

# =========================
# 3) churn 라벨 (프록시) 생성
# - last_played는 피처로 쓰지 않지만, 라벨 생성엔 사용
# - 리뷰 이후 90일 안에 last_played가 "없거나/그 이전이면" churn=1 로 가정
# =========================
df_label["last_played_dt"] = pd.to_datetime(df_label["last_played"], unit="s", errors="coerce")

# last_played가 NaT면 복귀 관측 안됨 -> churn=1 처리
df_label["churn"] = (
    df_label["last_played_dt"].isna()
    | (df_label["last_played_dt"] <= (df_label["review_dt"] + pd.Timedelta(days=90)))
).astype(int)

# =========================
# 4) 피처 선택 (사용자가 준 리스트 그대로)
# =========================
features = [
    "num_games_owned",
    "num_reviews_author",
    "deck_playtime_at_review",
    "voted_up",
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "review_length",
]

# 존재하는 컬럼만 사용 (실행 에러 방지)
features = [c for c in features if c in df_label.columns]


# 숫자형 강제 (문자 섞이면 터짐)
for c in features:
    df_label[c] = pd.to_numeric(df_label[c], errors="coerce")

# 결측은 0으로 (간단 버전). 더 정교하게 하려면 median 등으로 대체.
X = df_label[features].fillna(0)
y = df_label["churn"].astype(int)

# =========================
# 5) 시간 기준 Train/Valid Split (마지막 30일을 valid)
# =========================
split_date = LABEL_CUTOFF - pd.Timedelta(days=30)

train_mask = df_label["review_dt"] <= split_date
valid_mask = df_label["review_dt"] > split_date

X_train, y_train = X[train_mask], y[train_mask]
X_valid, y_valid = X[valid_mask], y[valid_mask]

print("Rows:", len(df_label), "| Train:", len(X_train), "| Valid:", len(X_valid))
print("Churn rate train:", round(y_train.mean(), 4), "| valid:", round(y_valid.mean(), 4))
print("Features used:", features)

# valid에 한 클래스만 있으면 AUC 계산이 안 됨
if y_valid.nunique() < 2:
    raise ValueError(f"Valid set에 클래스가 1개뿐입니다. (unique={y_valid.unique()}) split_date를 조정하거나 기간을 늘려야 합니다.")

# =========================
# 6) LightGBM 학습 + ROC-AUC
# =========================
model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc"
)

pred = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, pred)
print(f"Validation ROC-AUC: {auc:.4f}")

# =========================
# 7) (선택) 최근 90일(inference 구간) churn 확률 예측
# =========================
df_recent = df_180[df_180["review_dt"] > LABEL_CUTOFF].copy()

# 최근구간에도 동일 전처리 적용
df_recent["review_length"] = df_recent["review"].fillna("").astype(str).str.len()
if "deck_playtime_at_review" in df_recent.columns:
    df_recent["deck_playtime_at_review"] = df_recent["deck_playtime_at_review"].fillna(0)
for c in bool_cols:
    if c in df_recent.columns:
        df_recent[c] = df_recent[c].astype(int)
for c in features:
    df_recent[c] = pd.to_numeric(df_recent[c], errors="coerce")

X_recent = df_recent[features].fillna(0)
df_recent["churn_prob"] = model.predict_proba(X_recent)[:, 1]

df_recent[["steamid", "appid", "review_dt", "churn_prob"]].head()


Rows: 8176 | Train: 5690 | Valid: 2486
Churn rate train: 0.5232 | valid: 0.7285
Features used: ['num_games_owned', 'num_reviews_author', 'deck_playtime_at_review', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'review_length']
[LightGBM] [Info] Number of positive: 2977, number of negative: 2713
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000535 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 943
[LightGBM] [Info] Number of data points in the train set: 5690, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.523199 -> initscore=0.092861
[LightGBM] [Info] Start training from score 0.092861
Validation ROC-AUC: 0.5761


,steamid,appid,review_dt,churn_prob
0,76561199408691293,236390,2026-01-02 17:16:10,0.179399
1,76561199034590636,381210,2025-11-25 14:57:52,0.418319
6,76561199048280823,1142710,2025-10-25 20:21:14,0.414139
7,76561199099340178,3241660,2025-11-25 02:01:49,0.899405
14,76561199801588604,381210,2025-10-13 21:11:31,0.908785


In [30]:
# import pandas as pd

# fi = pd.Series(model.feature_importances_, index=features)\
#        .sort_values(ascending=False)
# print(fi)


In [31]:
# import lightgbm as lgb

# model = lgb.LGBMClassifier(
#     objective="binary",
#     n_estimators=300,
#     learning_rate=0.05,
#     num_leaves=31,
#     random_state=42
# )

# model.fit(
#     X_train, y_train,
#     eval_set=[(X_valid, y_valid)],
#     eval_metric="auc",
# )


In [32]:
# from sklearn.metrics import roc_auc_score

# valid_pred = model.predict_proba(X_valid)[:, 1]
# auc = roc_auc_score(y_valid, valid_pred)

# print(f"Validation ROC-AUC: {auc:.4f}")
